# Endothelial-cell PCA analysis

Project: **Characterisation of endothelial cells in progressive fatty liver disease in humans.**

This notebook loads a precomputed AnnData object and retains its PCA, neighbor-graph, Leiden, UMAP, feature-summary and ranking workflow. This notebook does not train a scVI model; model fitting is shown in the companion subset notebook.

These are edited source excerpts from my analysis notebooks. All saved outputs, execution counts, metadata, feature panels, group labels and findings have been removed. No data or figures are included. Private inputs are unavailable here, so these excerpts have not been rerun and do not reproduce the study.

Execution is disabled by default. Plotting calls remain as code only; any future output belongs in the private data directory. Algorithm settings are retained from the source and need reassessment for other inputs. Generic metadata keys stand in for private column names. Some routines use tutorial/shared analysis patterns, as in the original notebooks.

In [ ]:
from pathlib import Path

RUN_PRIVATE_ANALYSIS = False
RUN_EXTERNAL_ENRICHMENT = False
INPUT_FORMAT = "h5ad"
PRIVATE_DIR = Path("data/private")
OUTPUT_DIR = PRIVATE_DIR / "results"
BATCH_KEY = None  # Original scVI call did not supply a batch key.
selected_clusters = []  # Private, locally selected IDs; no study IDs supplied.
features_to_review = []  # Private feature panel omitted.
approved_enrichment_genes = []  # Only externally approved lists may be submitted.


## Imports, input and count layers

In [ ]:
if RUN_PRIVATE_ANALYSIS:
    # Required packages
    import numpy as np
    import pandas as pd
    import scanpy as sc
    import matplotlib as mpl
    import matplotlib.pyplot as plt
    import seaborn as sns
    from matplotlib import rcParams

    sc.settings.verbosity = 3             # verbosity: errors (0), warnings (1), info (2), hints (3)

    sc.settings.set_figure_params(scanpy=True, dpi=80, dpi_save=300, frameon=True, vector_friendly=True, fontsize=20)
    rcParams['figure.figsize'] = 8, 8
    plt.rcParams.update({'lines.markersize' : '15'})



    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    sc.settings.figdir = OUTPUT_DIR


In [ ]:
if RUN_PRIVATE_ANALYSIS:
    sc.settings.set_figure_params(scanpy=True, dpi=80, dpi_save=300, frameon=True, vector_friendly=True, fontsize=20)
    rcParams['figure.figsize'] = 8, 8
    import matplotlib.pyplot as plt
    plt.rcParams.update({'lines.markersize' : '15'})

    palette = ['#FFFF00', '#1CE6FF', '#FF34FF', '#FF4A46', '#008941', '#006FA6',
           '#A30059', '#FFDBE5', '#7A4900', '#0000A6', '#63FFAC', '#B79762',
           '#004D43', '#8FB0FF', '#997D87', '#5A0007', '#809693', '#6A3A4C',
           '#1B4400', '#4FC601', '#3B5DFF', '#4A3B53', '#FF2F80', '#61615A',
           '#BA0900', '#6B7900', '#00C2A0', '#FFAA92', '#FF90C9', '#B903AA',
           '#D16100', '#DDEFFF', '#000035', '#7B4F4B', '#A1C299', '#300018',
           '#0AA6D8', '#013349', '#00846F', '#372101', '#FFB500', '#C2FFED',
           '#A079BF', '#CC0744']


In [ ]:
if RUN_PRIVATE_ANALYSIS:
    adata = sc.read_h5ad(PRIVATE_DIR / "input_endothelial.h5ad")


In [ ]:
if RUN_PRIVATE_ANALYSIS:
    count_values = adata.X.data if hasattr(adata.X, "tocsr") else np.asarray(adata.X).ravel()
    if not (np.isfinite(count_values).all() and (count_values >= 0).all() and np.allclose(count_values, np.rint(count_values), rtol=0, atol=1e-8)):
        raise ValueError("Source normalization assumes raw counts in X; supply a raw-count object locally.")


In [ ]:
if RUN_PRIVATE_ANALYSIS:
    # ##Check QC- based on scanpy tutorial
    # # Broad QC feature prefixes, adjusted to the disclosed human context.
    # adata.var["mt"] = adata.var_names.str.startswith("MT-")
    # # ribosomal genes
    # adata.var["ribo"] = adata.var_names.str.startswith(("RPS", "RPL"))
    # # hemoglobin genes
    # adata.var["hb"] = adata.var_names.str.contains("^HB[^(P)]")

    # sc.pp.calculate_qc_metrics(adata, qc_vars=["mt", "ribo", "hb"], inplace=True, log1p=True)

    # sc.pl.violin(
    #     adata,
    #     ["pct_counts_mt", "pct_counts_ribo", "pct_counts_hb"],
    #     jitter=0.4,
    #     multi_panel=True,
    # )

    # sc.pl.violin(
    #     adata,
    #     ["n_genes_by_counts", "total_counts", "pct_counts_mt"],
    #     jitter=0.4,
    #     multi_panel=True,
    # )
    pass  # Optional QC recipe above is intentionally commented.


In [ ]:
if RUN_PRIVATE_ANALYSIS:
    # # 1. Backup raw counts if needed
    adata.layers["counts"] = adata.X.copy()

    # 2. Normalize total counts per cell to 10,000
    sc.pp.normalize_total(adata, target_sum=1e4)

    # 3. Log1p transform (log(x + 1))
    sc.pp.log1p(adata)

    # 4. Save the result to a new layer
    adata.layers["lognorm"] = adata.X.copy()

    # 5. (Optional) restore raw counts to X for compatibility with scVI
    #adata.X = adata.layers["counts"]


## HVGs, PCA, neighbors, Leiden and UMAP

The Seurat-v3 HVG call explicitly reads the count layer. PCA uses the log-normalized expression object. These settings are copied from the source without a claim that they are optimal.

In [ ]:
if RUN_PRIVATE_ANALYSIS:
    sc.pp.highly_variable_genes(adata, flavor="seurat_v3", layer="counts", n_top_genes=3000)
    # #adata  = adata[:, adata.var.highly_variable].copy()  # subsets hvgs only


In [ ]:
if RUN_PRIVATE_ANALYSIS:
    ###Using PCA instead of scvi
    sc.tl.pca(adata)
    sc.pl.pca_variance_ratio(adata, n_pcs=50, log=True)


In [ ]:
if RUN_PRIVATE_ANALYSIS:
    sc.pp.neighbors(adata, use_rep="X_pca", n_neighbors=31)  # try 10–50
    sc.tl.leiden(adata, resolution=0.8, flavor="igraph", n_iterations=2)
    sc.tl.umap(adata)  # min_dist ~0.1–0.5


In [ ]:
if RUN_PRIVATE_ANALYSIS:
    sc.pl.umap(adata, wspace=0.3, color=["leiden"], size=60, palette=palette)


## Parameterized expression summary

The source’s manually selected feature list is withheld. Supply `features_to_review` locally to use its mean/median ordering and alternative plotting options. No source pathway or feature selection is disclosed.

In [ ]:
if RUN_PRIVATE_ANALYSIS and features_to_review:
    import pandas as pd
    import numpy as np
    import matplotlib.pyplot as plt
    import scanpy as sc

    # ---- PARAMETERS — edit these ----
    genes_of_interest = features_to_review
    groupby    = "group_id"        # or 'cell_type', 'sample' etc.
    stat       = 'mean'        # 'median' or 'mean'
    ascending  = True            # True = ascending, False = descending
    plot_type  = 'dotplot'        # 'violin', 'stacked_violin', 'dotplot', 'matrixplot'

    # ---- FILTER TO GENES PRESENT ----
    valid_genes = [g for g in genes_of_interest if g in adata.var_names]
    missing     = [g for g in genes_of_interest if g not in adata.var_names]
    if missing:
        print(f"Not found in adata: {missing}")

    # ---- BUILD EXPRESSION DATAFRAME ----
    expr_df = pd.DataFrame(
        (adata[:, valid_genes].layers["lognorm"].toarray() if hasattr(adata[:, valid_genes].layers["lognorm"], "toarray")
         else np.asarray(adata[:, valid_genes].layers["lognorm"])),
        index=adata.obs_names,
        columns=valid_genes
    )
    expr_df['cluster'] = adata.obs[groupby].values

    # ---- COMPUTE STAT PER CLUSTER (aggregate across all valid genes) ----
    if stat == 'median':
        cluster_stat = expr_df.groupby('cluster')[valid_genes].median().mean(axis=1)
    else:
        cluster_stat = expr_df.groupby('cluster')[valid_genes].mean().mean(axis=1)

    cluster_order = cluster_stat.sort_values(ascending=ascending).index.tolist()
    print(f"Cluster order ({stat}, {'ascending' if ascending else 'descending'}):")
    print(cluster_order)

    # ================================================================
    # PLOT OPTIONS — uncomment the one you want
    # ================================================================

    # ---- OPTION 1: SCANPY VIOLIN ----
    if plot_type == 'violin':
        sc.pl.violin(
            adata,
            keys=valid_genes,
            groupby=groupby, layer="lognorm", use_raw=False,
            order=cluster_order,
            rotation=90,
            stripplot=False,       # cleaner without dots
            inner='box',           # shows median + IQR inside violin
            save="_private_analysis.pdf"
        )

    # ---- OPTION 2: STACKED VIOLIN (best for many genes) ----
    elif plot_type == 'stacked_violin':
        sc.pl.stacked_violin(
            adata,
            valid_genes,
            groupby=groupby, layer="lognorm", use_raw=False,
            order=cluster_order,
            swap_axes=False,
            row_palette='Reds',
            figsize=(14, len(valid_genes) * 0.6),
            save="_private_analysis.pdf"
        )

    # ---- OPTION 3: DOTPLOT (fraction + expression, very informative) ----
    elif plot_type == 'dotplot':
        sc.pl.dotplot(
            adata,
            valid_genes,
            groupby=groupby, layer="lognorm", use_raw=False,
            categories_order=cluster_order,
            standard_scale='var',
            cmap='Reds',
            save="_private_analysis.pdf"
        )

    # ---- OPTION 4: MATRIXPLOT (clean heatmap of mean expression) ----
    elif plot_type == 'matrixplot':
        sc.pl.matrixplot(
            adata,
            valid_genes,
            groupby=groupby, layer="lognorm", use_raw=False,
            categories_order=cluster_order,
            standard_scale='var',
            cmap='Reds',
            save="_private_analysis.pdf"
        )


## Exploratory marker ranking

These cell-level ranking routines retain tutorial and shared-analysis patterns from the source. They provide descriptive cluster summaries, not donor-level inferential tests. Feature names and ranking results are not included.

In [ ]:
if RUN_PRIVATE_ANALYSIS:
    #Tutorial based method
    # Obtain cluster-specific differentially expressed genes
    sc.tl.rank_genes_groups(adata, groupby="leiden", method="wilcoxon", layer="lognorm", use_raw=False)
    sc.pl.rank_genes_groups_dotplot(adata, groupby="leiden", layer="lognorm", use_raw=False, standard_scale="var", n_genes=20, save="_private_analysis.pdf")
    if selected_clusters:
        dc_cluster_genes = sc.get.rank_genes_groups_df(adata, group=selected_clusters[0]).head(20)["names"]
        sc.pl.umap(
            adata,
            color=[*dc_cluster_genes, "leiden"],
            layer="lognorm", use_raw=False,
            legend_loc="on data",
            frameon=False,
            ncols=3,
        )


In [ ]:
if RUN_PRIVATE_ANALYSIS:
    # Adapted shared-analysis pattern used in the original notebook.
    sc.tl.rank_genes_groups(adata, 'leiden', method='t-test', layer="lognorm", use_raw=False)
    sc.pl.rank_genes_groups(adata, n_genes=50, sharey=False, save="_private_analysis.pdf")

    result = adata.uns['rank_genes_groups']
    groups = result['names'].dtype.names
    adata_deg = pd.DataFrame(
        {group + '_' + key[:1]: result[key][group]
        for group in groups for key in ['names', 'pvals', 'scores']})
    adata_deg.to_csv(OUTPUT_DIR / "cluster_rankings.csv")


In [ ]:
if RUN_PRIVATE_ANALYSIS:
    # Get top 20 DEGs per cluster by score (score ranking; not fold-change ranking) similar to seurat in R
    result = adata.uns['rank_genes_groups']
    groups = result['names'].dtype.names

    # build a dataframe with names and scores
    deg_df = pd.DataFrame({
        'gene':    [gene  for group in groups for gene  in result['names'][group][:20]],
        'score':   [score for group in groups for score in result['scores'][group][:20]],
        'cluster': [group for group in groups for _    in result['names'][group][:20]]
    })

    # top 20 per cluster by score
    top20 = (
        deg_df
        .groupby('cluster')
        .apply(lambda x: x.nlargest(20, 'score'))
        .reset_index(drop=True)
    )

    top20_genes = top20['gene'].tolist()

    print(top20)
    print(f"\nTotal genes for heatmap: {len(top20_genes)}")

    # Visualize using plots in section below


## Ranked-feature visualizations

These additional plot recipes use the computed ranking rather than the source’s withheld hand-selected feature panels. Later dataset-specific interpretation and sample-comparison blocks have been excluded.

In [ ]:
if RUN_PRIVATE_ANALYSIS:
    # alternative — matrixplot (cleaner for many clusters)
    sc.pl.matrixplot(
        adata,
        var_names= top20_genes,
        groupby='leiden',
        layer='lognorm',
        use_raw=False,
        cmap='RdBu_r',
        standard_scale='var',
        figsize=(30, 10),
        save="_private_analysis.pdf"
    )


In [ ]:
if RUN_PRIVATE_ANALYSIS:
    # plot heatmap — equivalent to DoHeatmap in Seurat

    sc.pl.heatmap(
        adata,
        var_names=top20_genes,
        groupby='leiden',           # equivalent to grouping by cluster in Seurat
        layer='lognorm',            # use lognorm layer, same as Seurat's normalized data
        use_raw=False,
        show_gene_labels=True,
        standard_scale='var',
        figsize=(20, 10),           # match your pdf(width=40, height=50)
        save="_private_analysis.pdf"
    )
